# Blog 8 — Delta Lake Maintenance

## Complete Isolated Databricks Notebook

This notebook demonstrates:

- `OPTIMIZE`
- `ZORDER`
- `VACUUM`
- small-file problems
- data skipping
- Delta history
- retention and maintenance strategy

It creates its own isolated Unity Catalog schema and does not depend on DBFS, `/tmp`, or previous blogs.

**Catalog:** `workspace`  
**Schema:** `workspace.blog8_delta_maintenance`

## 1. Why Delta Lake Maintenance Matters

A Delta table can be logically correct but physically inefficient.

Too many small files can increase file discovery, metadata, scheduling, and I/O overhead.

```text
Logical correctness
        ≠
Physical efficiency
```

Maintenance improves physical organization without changing the logical meaning of the table.

## 2. The Three Core Operations

| Operation | Main purpose |
|---|---|
| `OPTIMIZE` | File compaction |
| `ZORDER` | Data locality / data skipping |
| `VACUUM` | Obsolete-file cleanup |

Mental model:

```text
OPTIMIZE → Compact
ZORDER   → Organize
VACUUM   → Clean
```

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import date

CATALOG = "workspace"
SCHEMA = "blog8_delta_maintenance"

ORDERS_TABLE = f"{CATALOG}.{SCHEMA}.orders"
SMALL_FILE_TABLE = f"{CATALOG}.{SCHEMA}.small_file_demo"

print("Catalog:", CATALOG)
print("Schema :", SCHEMA)
print("Spark  :", spark.version)

Catalog: workspace
Schema : blog8_delta_maintenance
Spark  : 4.1.0


## 3. Create the Dedicated Schema

The notebook resets only:

`workspace.blog8_delta_maintenance`

This makes the project rerunnable.

In [0]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")

spark.sql(f'''
CREATE SCHEMA {CATALOG}.{SCHEMA}
COMMENT 'Isolated Blog 8 - Delta Lake Maintenance'
''')

print(f"Created {CATALOG}.{SCHEMA}")

Created workspace.blog8_delta_maintenance


In [0]:
spark.sql(f"DESCRIBE SCHEMA EXTENDED {CATALOG}.{SCHEMA}").show(truncate=False)

+-------------------------+---------------------------------------------------------+
|database_description_item|database_description_value                               |
+-------------------------+---------------------------------------------------------+
|Catalog Name             |workspace                                                |
|Namespace Name           |blog8_delta_maintenance                                  |
|Comment                  |Isolated Blog 8 - Delta Lake Maintenance                 |
|Location                 |                                                         |
|Owner                    |bharath2704.a@gmail.com                                  |
|Properties               |                                                         |
|Predictive Optimization  |ENABLE (inherited from METASTORE metastore_aws_us_east_2)|
+-------------------------+---------------------------------------------------------+



## 4. Create the Main Orders Dataset

We create a deterministic dataset large enough to demonstrate Delta storage, filtering, optimization, and history.

The exact physical file count is intentionally not hard-coded because Spark/Databricks execution can produce different file layouts.

In [0]:
schema = StructType([
    StructField("order_id", LongType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("amount", DoubleType(), False),
    StructField("city", StringType(), True),
    StructField("status", StringType(), True)
])

cities = ["Chennai", "Bangalore", "Hyderabad", "Mumbai", "Delhi"]
statuses = ["COMPLETED", "PENDING", "CANCELLED"]

rows = [
    (
        i,
        (i % 10000) + 1,
        date(2026, 1, (i % 28) + 1),
        float((i % 500) + 50),
        cities[i % len(cities)],
        statuses[i % len(statuses)]
    )
    for i in range(1, 100001)
]

df = spark.createDataFrame(rows, schema)

print("Rows:", df.count())
df.printSchema()
display(df.limit(10))

Rows: 100000
root
 |-- order_id: long (nullable = false)
 |-- customer_id: integer (nullable = false)
 |-- order_date: date (nullable = false)
 |-- amount: double (nullable = false)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)



order_id,customer_id,order_date,amount,city,status
1,2,2026-01-02,51.0,Bangalore,PENDING
2,3,2026-01-03,52.0,Hyderabad,CANCELLED
3,4,2026-01-04,53.0,Mumbai,COMPLETED
4,5,2026-01-05,54.0,Delhi,PENDING
5,6,2026-01-06,55.0,Chennai,CANCELLED
6,7,2026-01-07,56.0,Bangalore,COMPLETED
7,8,2026-01-08,57.0,Hyderabad,PENDING
8,9,2026-01-09,58.0,Mumbai,CANCELLED
9,10,2026-01-10,59.0,Delhi,COMPLETED
10,11,2026-01-11,60.0,Chennai,PENDING


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable(ORDERS_TABLE)

print("Created:", ORDERS_TABLE)

spark.sql(f"DESCRIBE DETAIL {ORDERS_TABLE}")     .select("format", "numFiles", "sizeInBytes")     .show(truncate=False)

Created: workspace.blog8_delta_maintenance.orders
+------+--------+-----------+
|format|numFiles|sizeInBytes|
+------+--------+-----------+
|delta |8       |34052      |
+------+--------+-----------+



## 5. Understand the Small-File Problem

Small files can be caused by:

- frequent small writes
- micro-batches
- repeated appends
- excessive output partitioning
- poor ingestion design

The problem is not only storage size. File count also affects metadata, scheduling, discovery, and I/O overhead.

## 6. Create a Deliberate Small-File Demonstration

We perform many small append operations.

The resulting file count depends on execution behavior, so validation inspects actual metadata rather than assuming a fixed number.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {SMALL_FILE_TABLE}")

small_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("amount", DoubleType(), False)
])

for batch in range(20):
    small_rows = [
        (batch * 100 + i, (i % 100) + 1, float(i * 10))
        for i in range(100)
    ]

    spark.createDataFrame(small_rows, small_schema)         .write.format("delta")         .mode("append")         .saveAsTable(SMALL_FILE_TABLE)

small_before = spark.sql(
    f"DESCRIBE DETAIL {SMALL_FILE_TABLE}"
).select("format", "numFiles", "sizeInBytes").collect()[0]

print("Before OPTIMIZE")
print("Format:", small_before["format"])
print("Files :", small_before["numFiles"])
print("Bytes :", small_before["sizeInBytes"])

assert small_before["format"] == "delta"
assert small_before["numFiles"] >= 1

Before OPTIMIZE
Format: delta
Files : 20
Bytes : 29130


## 7. OPTIMIZE — File Compaction

`OPTIMIZE` rewrites files into a more efficient physical layout.

```text
Many small files
       ↓
    OPTIMIZE
       ↓
Fewer larger files
```

It is primarily a file-layout operation.

In [0]:
spark.sql(f"OPTIMIZE {SMALL_FILE_TABLE}")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
small_after = spark.sql(
    f"DESCRIBE DETAIL {SMALL_FILE_TABLE}"
).select("format", "numFiles", "sizeInBytes").collect()[0]

print("After OPTIMIZE")
print("Format:", small_after["format"])
print("Files :", small_after["numFiles"])
print("Bytes :", small_after["sizeInBytes"])

assert small_after["format"] == "delta"
assert spark.table(SMALL_FILE_TABLE).count() == 2000

print("Logical data preserved.")

After OPTIMIZE
Format: delta
Files : 1
Bytes : 2131
Logical data preserved.


## 8. OPTIMIZE Does Not Guarantee Every Query Becomes Fast

`OPTIMIZE` improves file organization.

Performance also depends on:

- predicates
- data volume
- data skipping
- joins
- cluster resources
- statistics
- workload

Therefore:

```text
OPTIMIZE ≠ automatic performance guarantee
```

## 9. Z-ORDER — Data Locality

Suppose a common workload is:

```sql
WHERE customer_id = 5000
```

For suitable workloads, Z-Ordering can improve physical locality for selected columns and therefore improve data-skipping opportunities.

In [0]:
query = f'''
SELECT COUNT(*)
FROM {ORDERS_TABLE}
WHERE customer_id = 5000
'''

display(spark.sql(query))

COUNT(*)
10


In [0]:
spark.sql(f"OPTIMIZE {ORDERS_TABLE} ZORDER BY (customer_id)")

print("Z-ORDER operation completed.")

Z-ORDER operation completed.


## 10. Z-Ordering Is Not Partitioning

**Partitioning** creates coarse directory-level separation.

**Z-Ordering** improves physical locality within the table for selected columns.

```text
Partitioning → coarse separation
Z-Ordering   → fine-grained locality
```

They are different techniques.

## 11. Data Skipping

Suppose file statistics indicate:

```text
customer_id = 1000 → 2000
```

and the query asks for:

```text
customer_id = 9000
```

The engine may determine the file cannot contain the value and skip it.

```text
Predicate
   ↓
File statistics
   ↓
Impossible?
   ↓
Skip file
```

Z-Ordering can improve physical organization for workloads where this matters.

## 12. Choose Z-Order Columns Carefully

Do not blindly Z-Order every column.

Choose based on:

- frequent filtering
- selectivity
- workload patterns
- table size
- maintenance cost

For this notebook, `customer_id` is the demonstration filter column.

## 13. Inspect the Query Plan

Use `EXPLAIN FORMATTED` to connect query predicates to the physical plan.

In [0]:
spark.sql(f'''
EXPLAIN FORMATTED
SELECT *
FROM {ORDERS_TABLE}
WHERE customer_id = 5000
''').show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 14. Proper Performance Validation

Do not claim Z-Ordering improved performance merely because the command succeeded.

A real comparison should examine:

- execution time
- files scanned
- bytes scanned
- query plan
- data-skipping behavior

Compare the same workload before and after maintenance.

In [0]:
display(
    spark.sql(f'''
    SELECT customer_id, COUNT(*) AS orders, SUM(amount) AS total_amount
    FROM {ORDERS_TABLE}
    WHERE customer_id = 5000
    GROUP BY customer_id
    ''')
)

customer_id,orders,total_amount
5000,10,5490.0


## 15. Delta Transaction History

Delta maintains transaction history.

Inspect it using `DESCRIBE HISTORY` to understand table versions and operations.

In [0]:
history = spark.sql(f"DESCRIBE HISTORY {ORDERS_TABLE}")
display(history.select("version", "timestamp", "operation"))

version,timestamp,operation
1,2026-08-24T08:46:58.000Z,OPTIMIZE
0,2026-08-24T08:45:55.000Z,CREATE OR REPLACE TABLE AS SELECT


## 16. VACUUM — Obsolete File Cleanup

Operations such as `UPDATE`, `DELETE`, `MERGE`, and `OPTIMIZE` can make older physical files obsolete.

`VACUUM` removes eligible obsolete files outside the configured retention period.

```text
Current + obsolete files
          ↓
       VACUUM
          ↓
Current files remain
Eligible obsolete files removed
```

## 17. VACUUM Is Not DELETE

```text
DELETE = logical data operation
VACUUM = physical storage cleanup
```

`DELETE` changes the rows visible in the table.

`VACUUM` removes obsolete physical files that are no longer needed under the retention policy.

## 18. VACUUM Retention and Safety

Retention protects files that may still be needed for:

- active readers
- time travel
- recovery workflows
- transactional safety

Do not casually use aggressive retention settings in production.

Retention should follow your organization's recovery and governance requirements.

## 19. Safe VACUUM Inspection

Where supported by the current Databricks runtime, `DRY RUN` allows inspection of eligible files without deleting them.

In [0]:
try:
    vacuum_result = spark.sql(f'''
        VACUUM {ORDERS_TABLE} RETAIN 168 HOURS DRY RUN
    ''')
    display(vacuum_result)
    print("VACUUM DRY RUN completed; no files were deleted.")
except Exception as e:
    print("VACUUM DRY RUN is not available in this environment.")
    print("No files were deleted.")
    print(str(e)[:1200])

path


VACUUM DRY RUN completed; no files were deleted.


## 20. Time Travel and Retention

Delta table versions can be inspected through history.

Older versions may require their underlying data files to remain available.

Therefore:

```text
VACUUM retention
       =
storage cleanup
+
historical availability policy
```

In [0]:
display(
    spark.sql(f"DESCRIBE HISTORY {ORDERS_TABLE}")
    .select("version", "timestamp", "operation")
)

version,timestamp,operation
1,2026-08-24T08:46:58.000Z,OPTIMIZE
0,2026-08-24T08:45:55.000Z,CREATE OR REPLACE TABLE AS SELECT


## 21. OPTIMIZE vs Z-ORDER vs VACUUM

| Operation | Purpose |
|---|---|
| `OPTIMIZE` | Compact small files |
| `ZORDER` | Improve locality for selected filter columns |
| `VACUUM` | Remove eligible obsolete physical files |
| `DELETE` | Logically remove rows |
| `UPDATE` | Logically modify rows |
| `MERGE` | Transactional insert/update/delete |

Mental model:

```text
OPTIMIZE → Compact
ZORDER   → Organize
VACUUM   → Clean
```

## 22. Production Maintenance Strategy

A practical pattern is:

```text
Ingestion
    ↓
Table grows
    ↓
Small-file accumulation
    ↓
OPTIMIZE when justified
    ↓
Query workload
    ↓
Z-ORDER where justified
    ↓
Retention window
    ↓
VACUUM
```

Frequency depends on ingestion rate, table size, file distribution, query patterns, retention requirements, and maintenance cost.

There is no universal "run every X hours" rule.

## 23. Don't Over-Maintain

Maintenance consumes compute.

A tiny table receiving a few rows per day usually does not justify aggressive optimization.

A continuously growing table with many small files may benefit substantially.

The engineering trade-off is:

```text
Maintenance cost
       VS
Performance / storage benefit
```

## 24. Optional Advanced Topic — Partitioning

Partitioning is related to physical organization but is not the main focus of this blog.

Poor partitioning can create:

- too many directories
- too many small files

High-cardinality columns such as `customer_id` are often poor partitioning choices.

Partitioning should be based on data distribution and workload rather than used automatically.

## 25. Full Validation

We validate:

- tables are Delta
- expected rows still exist
- OPTIMIZE preserved logical data
- transaction history exists
- Z-ORDER completed
- no unsafe VACUUM deletion was performed

In [0]:
orders_detail = spark.sql(
    f"DESCRIBE DETAIL {ORDERS_TABLE}"
).collect()[0]

small_detail = spark.sql(
    f"DESCRIBE DETAIL {SMALL_FILE_TABLE}"
).collect()[0]

assert orders_detail["format"] == "delta"
assert small_detail["format"] == "delta"

assert spark.table(ORDERS_TABLE).count() == 100000
assert spark.table(SMALL_FILE_TABLE).count() == 2000

assert spark.sql(f"DESCRIBE HISTORY {ORDERS_TABLE}").count() > 0

print("==============================================")
print("ALL BLOG 8 VALIDATION TESTS PASSED")
print("==============================================")

ALL BLOG 8 VALIDATION TESTS PASSED


## 26. Final Project Inspection

In [0]:
print("TABLES")
spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").show(truncate=False)

print("ORDERS DETAIL")
spark.sql(f"DESCRIBE DETAIL {ORDERS_TABLE}").show(truncate=False)

print("SMALL-FILE DEMO DETAIL")
spark.sql(f"DESCRIBE DETAIL {SMALL_FILE_TABLE}").show(truncate=False)

print("ORDERS HISTORY")
display(
    spark.sql(f"DESCRIBE HISTORY {ORDERS_TABLE}")
    .select("version", "timestamp", "operation")
)

print("SAMPLE DATA")
display(spark.table(ORDERS_TABLE).orderBy("order_id").limit(10))

TABLES
+-----------------------+---------------+-----------+
|database               |tableName      |isTemporary|
+-----------------------+---------------+-----------+
|blog8_delta_maintenance|orders         |false      |
|blog8_delta_maintenance|small_file_demo|false      |
+-----------------------+---------------+-----------+

ORDERS DETAIL
+------+------------------------------------+----------------------------------------+-----------+--------+----------------------+-------------------+----------------+-----------------+--------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                                    |description|location|createdAt     

version,timestamp,operation
1,2026-08-24T08:46:58.000Z,OPTIMIZE
0,2026-08-24T08:45:55.000Z,CREATE OR REPLACE TABLE AS SELECT


SAMPLE DATA


order_id,customer_id,order_date,amount,city,status
1,2,2026-01-02,51.0,Bangalore,PENDING
2,3,2026-01-03,52.0,Hyderabad,CANCELLED
3,4,2026-01-04,53.0,Mumbai,COMPLETED
4,5,2026-01-05,54.0,Delhi,PENDING
5,6,2026-01-06,55.0,Chennai,CANCELLED
6,7,2026-01-07,56.0,Bangalore,COMPLETED
7,8,2026-01-08,57.0,Hyderabad,PENDING
8,9,2026-01-09,58.0,Mumbai,CANCELLED
9,10,2026-01-10,59.0,Delhi,COMPLETED
10,11,2026-01-11,60.0,Chennai,PENDING


## 27. Production Checklist

- [x] Understand small-file problem
- [x] Demonstrate multiple small writes
- [x] Inspect Delta metadata
- [x] Use `OPTIMIZE`
- [x] Validate logical data after compaction
- [x] Understand Z-Ordering
- [x] Apply Z-Ordering to a justified filter column
- [x] Understand data skipping
- [x] Inspect query plan
- [x] Understand `VACUUM`
- [x] Understand retention
- [x] Safely inspect VACUUM eligibility when supported
- [x] Inspect Delta history
- [x] Understand time-travel implications
- [x] Distinguish DELETE from VACUUM
- [x] Understand maintenance trade-offs
- [x] Use an isolated Unity Catalog schema
- [x] Avoid DBFS and `/tmp`

## 28. Final Mental Model

```text
                    DELTA TABLE
                         │
              ┌──────────┼──────────┐
              ↓          ↓          ↓
          OPTIMIZE     Z-ORDER    VACUUM
              │          │          │
              ↓          ↓          ↓
           Compact    Organize    Clean
            files      locality   obsolete
                                   files
              │          │          │
              └──────────┼──────────┘
                         ↓
                HEALTHIER TABLE
```

### Remember

> **OPTIMIZE = compact**

> **Z-ORDER = organize for filtering**

> **VACUUM = clean obsolete files**

## 29. Blog 8 Complete

We have moved from:

> How do we safely change a Delta table's schema?

to:

> How do we keep a growing Delta table physically healthy and maintainable?

### Next

# Blog 9 — Failure Recovery

```text
Pipeline Failure
      ↓
Partial Processing
      ↓
Retries
      ↓
Idempotency
      ↓
Delta Transactions
      ↓
Checkpoints / State
      ↓
Recovery
      ↓
Audit + Monitoring
```